In [ ]:
import os
import sys
import urllib.request
import datetime
import time
import json
import urllib.parse

In [ ]:
from google.colab import userdata
client_id = userdata.get('naver_client_id')
client_secret = userdata.get('naver_client_secret')

**1. API 요청 및 응답 수신**

네이버 API 서버에 접속하기 위해 인증 헤더를 설정하고, 서버로부터 응답 데이터를 받아오는 가장 기초적인 함수

In [ ]:
## [CODE 1]
def getRequestUrl(url):
    req = urllib.request.Request(url)
    # API 권한 인증을 위한 Client ID와 Secret을 요청 헤더에 추가
    req.add_header("X-Naver-Client-Id", client_id)
    req.add_header("X-Naver-Client-Secret", client_secret)

    try:
        response = urllib.request.urlopen(req)
        # 요청이 성공(HTTP 200 OK)하면 성공 메시지를 출력하고 데이터를 utf-8로 디코딩하여 반환
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')

    except Exception as e:
        # 접속 에러 발생 시 에러 내용과 해당 URL을 출력하여 문제 파악을 도움
        print(e)
        print("[%s] Error for URL : %s" % (datetime.datetime.now(), url))
        return None

**2. 검색 URL 생성 및 호출**

사용자가 입력한 검색어와 조건(시작 위치, 노출 개수)을 조합하여 API 호출 규격에 맞는 URL을 생성하고 결과를 JSON으로 변환

In [ ]:
## [CODE 2]
def getNaverSearch(node, srcText, start, display):
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    # 검색어는 URL 인코딩(quote) 처리하여 파라미터로 구성
    parameters = "?query=%s&start=%s&display=%s" % (urllib.parse.quote(srcText), start, display)

    url = base + node + parameters
    responseDecode = getRequestUrl(url)  # [CODE 1] 함수를 통해 데이터 수신

    if (responseDecode == None):
        return None
    else:
        # 수신된 JSON 문자열을 파이썬의 딕셔너리/리스트 객체로 변환
        return json.loads(responseDecode)

**3. 필요한 데이터 항목 추출**

API 응답 결과에는 불필요한 정보도 포함되어 있기에 제목, 설명, 링크, 날짜 등 분석에 꼭 필요한 정보만 골라내어 정리

In [ ]:
## [CODE 3]
def getPostData(post, jsonResult, cnt):
    title = post['title']
    description = post['description']
    org_link = post['originallink']
    link = post['link']

    # 제공되는 날짜 문자열 형식을 분석하여 파이썬의 표준 날짜 형식으로 변환
    pData = datetime.datetime.strptime(post['pubDate'], '%a, %d %b %Y %H:%M:%S +0900')
    pDate = pData.strftime('%Y-%m-%d %H:%M:%S')

    # 추출한 데이터를 정해진 구조(딕셔너리)에 맞춰 리스트에 누적
    jsonResult.append({'cnt':cnt, 'title':title, 'description':description, 'org_link':org_link, 'link':link, 'pDate':pDate})

**4. 크롤링 전체 프로세스 제어 및 저장**

사용자로부터 검색어를 입력받고, 네이버 뉴스 정책(최대 1,000건)에 맞춰 반복 수집을 진행한 뒤 최종 결과를 JSON 파일로 저장

In [ ]:
## [CODE 0]
def main():
    node = 'news'  # 검색 카테고리 설정
    srcText = input('검색어를 입력하세요: ')
    cnt = 0
    jsonResult = []

    # API 검색 수행 (최대 100개 단위로 요청)
    jsonResponse = getNaverSearch(node, srcText, 1, 100)
    total = jsonResponse['total']

    # 데이터가 더 있고, 네이버 제한(1,000건) 이내라면 계속해서 수집 반복
    while ((jsonResponse != None) and (jsonResponse['display'] != 0)):
        for post in jsonResponse['items']:
            cnt += 1
            getPostData(post, jsonResult, cnt)

        # 다음 수집 시작 위치 업데이트
        start = jsonResponse['start'] + jsonResponse['display']
        if start > 1000: break  # 네이버는 1000개까지만 조회를 허용함
        jsonResponse = getNaverSearch(node, srcText, start, 100)

    print('전체 검색: %d 건' % total)

    # 수집 완료된 데이터를 JSON 파일로 깔끔하게 저장 (한글 깨짐 방지 처리)
    with open('%s_naver_%s.json' % (srcText, node), 'w', encoding='utf8') as outfile:
        jsonFile = json.dumps(jsonResult, indent=4, sort_keys=True, ensure_ascii=False)
        outfile.write(jsonFile)

    print("가져온 데이터 : %d 건" % (cnt))
    print('%s_naver_%s.json SAVED' % (srcText, node))

if __name__ == '__main__':
    main()

검색어를 입력하세요: 제미나이
[2025-11-28 12:22:00.400425] Url Request Success
[2025-11-28 12:22:01.362056] Url Request Success
[2025-11-28 12:22:02.328920] Url Request Success
[2025-11-28 12:22:03.271380] Url Request Success
[2025-11-28 12:22:04.251351] Url Request Success
[2025-11-28 12:22:05.257557] Url Request Success
[2025-11-28 12:22:06.534565] Url Request Success
[2025-11-28 12:22:07.534486] Url Request Success
[2025-11-28 12:22:08.564842] Url Request Success
[2025-11-28 12:22:09.564871] Url Request Success
전체 검색: 32300 건
가져온 데이터 : 1000 건
제미나이_naver_news.json SAVED
